# Method 2 -- Supervised Fine-Tuning of DeBERTa-v3-small

**Root-cause note:** earlier attempts (see git history / `docs/report.pdf` Section 4.3) reported the fine-tuned model collapsing to predicting a single class, with loss stuck near `ln(2) = 0.693`. A raw-PyTorch diagnostic (same learning rate, same data pipeline, no `Trainer`) converged normally within under one epoch on 1,000 examples, which ruled out truncation, label polarity, tokenization, and model capacity as causes. The actual culprit was a custom `WeightedTrainer` applying inverse-class-frequency loss weighting on an already-balanced dataset (5,010 vs 4,990) combined with label smoothing and a cosine schedule -- the notebook's own comments record that the weighting "caused the model to always predict Class 1." `src/models/fine_tune.py` removes that custom loss and uses the `Trainer`'s default (plain, unweighted) cross-entropy, which this notebook demonstrates converges correctly.

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_halueval_qa
from src.models.fine_tune import fine_tune
from src.evaluation.metrics import print_report
from src.data.utils import set_seed

set_seed(42)
ds = load_halueval_qa(num_samples=10_000)

## 80/20 split

In [ ]:
trainer_8020, test_8020, _ = fine_tune(ds, scheme='80/20', num_train_epochs=3)
results_8020 = trainer_8020.evaluate(test_8020)
results_8020

## 70/15/15 split

In [ ]:
trainer_701515, test_701515, _ = fine_tune(ds, scheme='70/15/15', num_train_epochs=3)
results_701515 = trainer_701515.evaluate(test_701515)
results_701515

## Prediction distribution check (confirms the model is no longer degenerate)

In [ ]:
import numpy as np
from collections import Counter

preds = trainer_701515.predict(test_701515)
pred_labels = np.argmax(preds.predictions, axis=1)
print('Prediction distribution:', Counter(pred_labels.tolist()))
print('True distribution:', Counter(test_701515['labels']))

In [ ]:
with open('../outputs/tables/finetune_results.json', 'w') as f:
    json.dump({
        'DeBERTa-v3-small (Fine-Tuned, 80/20)': {k.replace('eval_', ''): v for k, v in results_8020.items() if k.startswith('eval_')},
        'DeBERTa-v3-small (Fine-Tuned, 70/15/15)': {k.replace('eval_', ''): v for k, v in results_701515.items() if k.startswith('eval_')},
    }, f, indent=2)